In [52]:
# modules
import pandas
import numpy
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LassoCV
from sklearn.model_selection import RepeatedKFold

from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

In [16]:
rentData = pandas.read_csv("housing_train.csv")
# combine state and region data
rentData["stateRegion"] = rentData["state"] + " " + rentData["region"]
# drop columns that are not needed
rentData.drop(['url', 'region_url', 'description', 'lat', 'long', 'image_url', 'region', 'state'], axis=1, inplace=True)


In [20]:
rentData.head(5)

,id,price,type,sqfeet,beds,baths,cats_allowed,dogs_allowed,smoking_allowed,wheelchair_access,electric_vehicle_charge,comes_furnished,laundry_options,parking_options,stateRegion
0,7039061606,1195,apartment,1908,3,2.0,1,1,1,0,0,0,laundry on site,street parking,al birmingham
1,7041970863,1120,apartment,1319,3,2.0,1,1,1,0,0,0,laundry on site,off-street parking,al birmingham
2,7041966914,825,apartment,1133,1,1.5,1,1,1,0,0,0,laundry on site,street parking,al birmingham
3,7041966936,800,apartment,927,1,1.0,1,1,1,0,0,0,laundry on site,street parking,al birmingham
4,7041966888,785,apartment,1047,2,1.0,1,1,1,0,0,0,laundry on site,street parking,al birmingham


In [ ]:
# turn categorical data into dummy variables
# drop first for all dummy variables to get to k-1 columns, and set type to float
geo_dummies = pandas.get_dummies(rentData['stateRegion'], drop_first=True, dtype=float)
geo_dummies = geo_dummies.add_prefix('stateRegion_')
type_dummies = pandas.get_dummies(rentData['type'], drop_first=True, dtype=float)
type_dummies = type_dummies.add_prefix('type_')
laundry_dummies = pandas.get_dummies(rentData['laundry_options'], drop_first=True, dtype=float)
laundry_dummies = laundry_dummies.add_prefix('laundry_')
parking_dummies = pandas.get_dummies(rentData['parking_options'], drop_first=True, dtype=float)
parking_dummies = parking_dummies.add_prefix('parking_')


In [26]:
parking_dummies.head(5)

,parking_carport,parking_detached garage,parking_no parking,parking_off-street parking,parking_street parking,parking_valet parking
0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0,1.0,0.0
3,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,0.0,0.0,0.0,1.0,0.0


In [38]:
yData = rentData.price
dataToStandardize = rentData[["sqfeet", "beds", "baths"]]

In [39]:
# Initialize the scaler
scaler = StandardScaler()
# Fit and transform the data
standardizedData = scaler.fit_transform(dataToStandardize)
# reapply indexes 
standardizedDataFrame = pandas.DataFrame(standardizedData, columns=["sqfeet", "beds", "baths"], index=dataToStandardize.index)
standardizedDataFrame.head(10)

,sqfeet,beds,baths
0,0.035300,0.294588,0.819622
1,0.009767,0.294588,0.819622
2,0.001705,-0.247140,0.026232
3,-0.007225,-0.247140,-0.767158
4,-0.002023,0.023724,-0.767158
5,0.008857,0.023724,0.819622
6,0.011111,0.023724,0.819622
7,0.052639,0.294588,0.819622
8,0.002702,0.294588,0.819622
9,-0.016935,-0.247140,-0.767158


In [40]:
xData = rentData[["cats_allowed", "dogs_allowed", "smoking_allowed", "wheelchair_access", "electric_vehicle_charge", "comes_furnished"]]

In [41]:
# join all data together for model
xData = xData.join([standardizedDataFrame, geo_dummies, type_dummies, laundry_dummies, parking_dummies])


In [42]:
xData

,cats_allowed,dogs_allowed,smoking_allowed,wheelchair_access,electric_vehicle_charge,comes_furnished,sqfeet,beds,baths,stateRegion_ak fairbanks,...,laundry_laundry on site,laundry_no laundry on site,laundry_w/d hookups,laundry_w/d in unit,parking_carport,parking_detached garage,parking_no parking,parking_off-street parking,parking_street parking,parking_valet parking
0,1,1,1,0,0,0,0.035300,0.294588,0.819622,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1,1,1,0,0,0,0.009767,0.294588,0.819622,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,1,1,1,0,0,0,0.001705,-0.247140,0.026232,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,1,1,1,0,0,0,-0.007225,-0.247140,-0.767158,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,1,1,1,0,0,0,-0.002023,0.023724,-0.767158,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
265185,1,1,1,0,0,0,-0.001417,0.023724,0.819622,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
265186,1,1,1,0,0,0,-0.003194,0.023724,0.026232,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
265187,1,1,1,0,0,0,0.024549,0.023724,0.026232,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
265188,1,1,1,0,0,0,0.005476,0.294588,0.026232,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [43]:
# create train / test split
X_train, X_test, y_train, y_test = train_test_split(xData, yData, test_size=0.2) 

In [46]:
#define cross-validation method to evaluate model
cv = RepeatedKFold(n_splits=10, n_repeats=3, random_state=1)

#define model
model = LassoCV(alphas=numpy.arange(0, 1, 0.01), cv=cv, n_jobs=-1)

#fit model
model.fit(X_train, y_train)

C:\Users\Ryan\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_coordinate_descent.py:683: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\Ryan\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_coordinate_descent.py:683: UserWarning: Coordinate descent without L1 regularization may lead to unexpected results and is discouraged. Set l1_ratio > 0 to add L1 regularization.
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\Ryan\AppData\Roaming\Python\Python311\site-packages\sklearn\linear_model\_coordinate_descent.py:683: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations. Duality gap: 788107253929336.9, tolerance: 158355341735.8574
  model = cd_fast.enet_coordinate_descent_gram(
C:\Users\Ryan\AppData\Roaming\Python\Python311\site-packages

LassoCV(alphas=array([0.  , 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1 ,
       0.11, 0.12, 0.13, 0.14, 0.15, 0.16, 0.17, 0.18, 0.19, 0.2 , 0.21,
       0.22, 0.23, 0.24, 0.25, 0.26, 0.27, 0.28, 0.29, 0.3 , 0.31, 0.32,
       0.33, 0.34, 0.35, 0.36, 0.37, 0.38, 0.39, 0.4 , 0.41, 0.42, 0.43,
       0.44, 0.45, 0.46, 0.47, 0.48, 0.49, 0.5 , 0.51, 0.52, 0.53, 0.54,
       0.55, 0.56, 0.57, 0.58, 0.59, 0.6 , 0.61, 0.62, 0.63, 0.64, 0.65,
       0.66, 0.67, 0.68, 0.69, 0.7 , 0.71, 0.72, 0.73, 0.74, 0.75, 0.76,
       0.77, 0.78, 0.79, 0.8 , 0.81, 0.82, 0.83, 0.84, 0.85, 0.86, 0.87,
       0.88, 0.89, 0.9 , 0.91, 0.92, 0.93, 0.94, 0.95, 0.96, 0.97, 0.98,
       0.99]),
        cv=RepeatedKFold(n_repeats=3, n_splits=10, random_state=1), n_jobs=-1)

In [47]:
# display lambda value
print(model.alpha_)

0.99


In [50]:
model = Lasso(alpha=0.99)
model.fit(X_train, y_train)

Lasso(alpha=0.99)

In [51]:
model.score(X_test, y_test)

-7.83285946137306

In [53]:
model = sm.OLS(y_train, X_train).fit()
print(model.summary())

                                 OLS Regression Results                                
Dep. Variable:                  price   R-squared (uncentered):                   0.001
Model:                            OLS   Adj. R-squared (uncentered):             -0.001
Method:                 Least Squares   F-statistic:                             0.5185
Date:                Sat, 07 Mar 2026   Prob (F-statistic):                        1.00
Time:                        20:36:32   Log-Likelihood:                     -3.6124e+06
No. Observations:              212152   AIC:                                  7.226e+06
Df Residuals:                  211816   BIC:                                  7.229e+06
Df Model:                         336                                                  
Covariance Type:            nonrobust                                                  
                                                coef    std err          t      P>|t|      [0.025      0.975]
----------